# NEX-GDDP-CMIP6 Dataset Ingestion (metadata stated by hand)

This is `nexgddp_cmip6_ingest.ipynb` with the **probe call replaced by fields written out in
Python**. It is the same dataset, the same registration and the same verification; the only
thing that changed is where the metadata comes from. `MdGdalSource` therefore has to work
without `/dataset/probe-md` — which is the point of this notebook.

Everything stated below is a property of the NEX-GDDP-CMIP6 dataset that the importer
already knows: one daily year per file, a 0.25 degree 1440x600 grid wrapping 0..360, CF
`time` in `days since 1850-01-01` stamped at noon. Nothing is read from a file. The values
were checked against the real data before being written here — see the notes at each
constant.

The companion notebook keeps the probe version, so a reviewer can diff the two
`register_dataset` implementations and see exactly what the probe saves. The probe branch
also fixes the single-array band name, so the two agree once it lands.


In [ ]:
import datetime as dt
import json
import re
from pathlib import Path

import fsspec
import numpy as np
import s3fs

import geoengine as ge
import geoengine_api_client as geoc


## Configuration

Update these paths to match your environment.


In [ ]:
# NEX-GDDP-CMIP6 S3 bucket (public). Files are registered as external data, so GeoEngine
# streams them through GDAL's /vsicurl rather than copying them onto a volume.
S3_BUCKET = "nex-gddp-cmip6"
S3_REGION = "us-west-2"
S3_PREFIX = "NEX-GDDP-CMIP6"

# GeoEngine connection
GEOENGINE_URL = "http://localhost:3030/api"
GEOENGINE_USER = "admin@localhost"
GEOENGINE_PASSWORD = "adminadmin"

# Full model registry: the authoritative id/variant/grid/scenario list. The dev registry
# (`nexgddp_models.toml`) is trimmed to what is ingested, so ingest reads this one and the
# trimmed file is regenerated from it afterwards.
REGISTRY_PATH = Path("../backend/conf/nexgddp_models.full.toml")

try:
    import tomllib
except ModuleNotFoundError:  # Python < 3.11
    import tomli as tomllib

with open(REGISTRY_PATH, "rb") as fh:
    REGISTRY = tomllib.load(fh)["model"]

MODELS = [{"id": m["id"], "variant": m["variant"], "grid": m["grid"]} for m in REGISTRY]

SCENARIOS = sorted({s for m in REGISTRY for s in m["scenarios"]})

# Variables. Each one lives in its own set of yearly files, so each becomes its own dataset
# with a single band and z = time.
VARIABLES = ["tasmax", "tasmin", "pr"]

# Variable metadata. `long_name` and `unit` are the CF attributes the files carry - the
# probe would have used exactly these for the band, so they are stated here rather than
# derived, and the band is what a reader sees in the layer listing.
VARIABLE_CONFIGS = {
    "tasmax": {
        "name": "Maximum Temperature",
        "long_name": "Daily Maximum Near-Surface Air Temperature",
        "measurement": "temperature",
        "unit": "K",
        "min": 200,
        "max": 340,
        "color_map": "coolwarm",
    },
    "tasmin": {
        "name": "Minimum Temperature",
        "long_name": "Daily Minimum Near-Surface Air Temperature",
        "measurement": "temperature",
        "unit": "K",
        "min": 200,
        "max": 340,
        "color_map": "coolwarm",
    },
    "pr": {
        "name": "Precipitation",
        "long_name": "Precipitation",
        "measurement": "precipitation",
        "unit": "kg m-2 s-1",
        "min": 0,
        "max": 0.01,
        "color_map": "Blues",
    },
}

# Number of daily slices read per request. A slice of this grid is 600x1440 float32 = 3.4 MB,
# so a batch is one IPC message: 4 is ~14 MB, 16 would be ~55 MB and ~22 s cold over the
# network. Queries for fewer days than this still read only the days they ask for.
#
# This is a *dataset* property (`maxZBatchSize`), not an operator parameter: a batch is sized
# against the slice size of the data, so whoever registers the dataset decides it and every
# consumer workflow inherits it.
MAX_Z_BATCH_SIZE = 4

# GDAL must be allowed to read .nc over /vsicurl.
VSICURL_ALLOWED_EXTENSIONS = ".tif,.tiff,.TIF,.jp2,.ovr,.nc"

# ponytail: trial mode keeps AWS traffic low. Set DEV_ONLY = False for the full matrix
# (all 5 models x 3 scenarios x 3 variables).
#
# Registration no longer opens every file, so this is no longer a wall-clock cost of probing -
# but the *verification* still reads a real window per dataset, and a full run reads 45 of
# them. DEV_ONLY therefore still trims the matrix and caps the years per scenario.
# `YEARS_PER_SCENARIO` is None for a full run.
#
# 3 years gives two file *boundaries* per scenario, which is the minimum that still proves
# file concatenation: a read window straddling a boundary spans two files in one batch.
YEARS_PER_SCENARIO = 5
DEV_ONLY = True
if DEV_ONLY:
    MODELS = MODELS[:1]
    SCENARIOS = SCENARIOS[:2]      # historical + ssp245
    VARIABLES = VARIABLES[:1]      # tasmax
else:
    YEARS_PER_SCENARIO = None


## Initialize GeoEngine Connection


In [ ]:
ge.initialize(
    GEOENGINE_URL,
    credentials=(GEOENGINE_USER, GEOENGINE_PASSWORD),
)

# The files stay on S3; `external` registers them by reference.
DATA_STORE = "external"

permission_tuples = [
    (ge.permissions.ANONYMOUS_USER_ROLE_ID, ge.Permission.READ),
    (ge.permissions.REGISTERED_USER_ROLE_ID, ge.Permission.READ),
]


## List Files

For each model/scenario/variable combination, list the yearly NetCDF files. Nothing is
probed: the year in each file name is the one fact taken from the files, and it is what
decides the file's time axis.

That is one S3 listing per combination and no GDAL at all - which is the difference between
this notebook and its probe companion.


In [ ]:
def list_netcdf_files(model_id, scenario, variable, variant, grid):
    """List all yearly NetCDF files for a model/scenario/variable combination."""
    fs = s3fs.S3FileSystem(anon=True)

    # Determine year range based on scenario
    if scenario == "historical":
        years = range(1950, 2015)
    else:
        years = range(2015, 2101)

    files = []
    for year in years:
        # NEX-GDDP-CMIP6 published corrected `{year}_v2.0.nc` copies for early
        # historical years; prefer them over the uncorrected base file.
        base = (f"{S3_BUCKET}/{S3_PREFIX}/{model_id}/{scenario}/{variant}/{variable}/"
                f"{variable}_day_{model_id}_{scenario}_{variant}_{grid}_")
        path = next(
            (base + cand for cand in (f"{year}_v2.0.nc", f"{year}.nc")
             if fs.exists(base + cand)),
            None,
        )
        if path is not None:
            files.append(path)
        else:
            print(f"Warning: File not found: {base}{year}.nc")
        # a dev run stops after the cap; a full run keeps every year
        if YEARS_PER_SCENARIO is not None and len(files) >= YEARS_PER_SCENARIO:
            break

    return sorted(files)


def vsicurl_url(bucket_key):
    """The /vsicurl path GeoEngine opens for a bucket key.

    s3fs keys start with the bucket name (`nex-gddp-cmip6/NEX-GDDP-CMIP6/...`), but the
    bucket is already in the hostname of an S3 URL, so the key must not repeat it --
    doing so yields a 404 from GDAL.
    """
    bucket, key = bucket_key.split("/", 1)
    return f"/vsicurl/https://{bucket}.s3.{S3_REGION}.amazonaws.com/{key}"


def collect_targets():
    """Every (model, scenario, variable) combination that has files on S3."""
    targets = []
    for model in MODELS:
        for scenario in SCENARIOS:
            for variable in VARIABLES:
                files = list_netcdf_files(
                    model["id"], scenario, variable, model["variant"], model["grid"],
                )
                if not files:
                    print(f"No files found, skipping {model['id']}/{scenario}/{variable}")
                    continue
                targets.append((model, scenario, variable, files))
    return targets


TARGETS = collect_targets()
for model, scenario, variable, files in TARGETS:
    print(f"{model['id']} / {scenario} / {variable}: {len(files)} yearly files")


## Declare the Dataset Metadata

`declare_md_dataset` states every field the probe would have derived. Three of them are easy
to get wrong, so they are spelled out:

1. **Two geo transforms, and they differ.** `params.geoTransform` is the grid *as stored*
   (CF `lat` ascends, so `y_pixel_size` is positive and the array is south-up);
   `resultDescriptor.spatialGrid` is the grid *as presented* (always north-up, longitudes
   re-labelled -180..180 because the stored coverage wraps 0..360). Both origins are the
   first cell's **edge**, not its centre: `lon` centres start at 0.125, so the edge is
   `0.125 - 0.25/2 = 0.0`.

2. **The time values are stamped at noon.** CF stores daily samples at the cell centre, so
   the files' `time` is `days since 1850-01-01` with a `.5` offset: the 1950 file's first
   value is 36524.5, i.e. `1950-01-01T12:00Z`. An interval is therefore **noon to noon**,
   not midnight to midnight. Slice j is day j of the year either way.

3. **`timeSteps` is one interval per z slice**, not one for the whole file, and
   `timeDescriptor` must reproduce them exactly - the server compares them and rejects a row
   where they disagree.


In [ ]:
def build_symbology(variable):
    config = VARIABLE_CONFIGS[variable]
    return ge.RasterSymbology(
        raster_colorizer=ge.SingleBandRasterColorizer(
            band=0,
            band_colorizer=ge.Colorizer.linear_with_mpl_cmap(
                color_map=config["color_map"],
                min_max=(config["min"], config["max"]),
                n_steps=10,
            ),
        ),
    )


# --- the dataset's grid ----------------------------------------------------
#
# 0.25 degree cells, 1440 x 600, covering lon 0..360 and lat -60..90.
N_LON, N_LAT = 1440, 600
DEG = 0.25

# The stored grid: `lat` ascends (-59.875 .. 89.875), so the array is south-up and
# `y_pixel_size` is positive. Origins are cell edges.
STORED_GEO_TRANSFORM = geoc.GeoTransform(
    origin_coordinate=geoc.Coordinate2D(x=0.0, y=-60.0),
    x_pixel_size=DEG,
    y_pixel_size=DEG,
)

# The presented grid: north-up, longitudes shifted to -180..180 (the stored coverage wraps
# 0..360). `-60.0 + N_LAT * DEG` is the north edge.
PRESENTED_GEO_TRANSFORM = geoc.GeoTransform(
    origin_coordinate=geoc.Coordinate2D(x=-180.0, y=90.0),
    x_pixel_size=DEG,
    y_pixel_size=-DEG,
)

# The presented footprint of every row, in spatial coordinates.
PRESENTED_FOOTPRINT = geoc.SpatialPartition2D(
    upperLeftCoordinate=geoc.Coordinate2D(x=-180.0, y=90.0),
    lowerRightCoordinate=geoc.Coordinate2D(x=180.0, y=-60.0),
)

DAY_MS = 86_400_000
NOON_MS = 12 * 3_600_000


def file_year(key):
    """The calendar year a yearly NEX-GDDP file covers, from its name.

    The name ends in `_{year}.nc` or `_{year}_v2.0.nc`. This is the one fact taken from the
    file name rather than known from the dataset's convention.
    """
    match = re.search(r"_(\d{4})(?:_v\d+\.\d+)?\.nc$", key)
    if match is None:
        raise ValueError(f"no year in file name: {key}")
    return int(match.group(1))


def year_slice_start_ms(year):
    """ms of `year-01-01T12:00:00Z`: the file's first time value is *noon*, not midnight.

    Checked against the data: the 1950 file's first value is 36524.5 and
    `1850-01-01T00:00Z + 36524.5 days` is `1950-01-01T12:00Z`.
    """
    return int(dt.datetime(year, 1, 1, tzinfo=dt.timezone.utc).timestamp() * 1000) + NOON_MS


def file_time_steps(year):
    """One interval per daily slice of the file covering `year`, noon to noon.

    Slice j of that file is day j of the year. `n_days` follows the calendar (366 in a leap
    year), and the server checks it against the array's z size when the row is added - so a
    misread year fails at registration, not at query time.
    """
    n_days = (dt.date(year + 1, 1, 1) - dt.date(year, 1, 1)).days
    start = year_slice_start_ms(year)
    return [
        geoc.TimeInterval(start=start + j * DAY_MS, end=start + (j + 1) * DAY_MS)
        for j in range(n_days)
    ]


def daily_descriptor(start_ms, end_ms):
    """The descriptor the server derives from `file_time_steps`, stated the same way.

    It is compared with the steps *exactly*, so it has to be `Regular(origin=first start,
    step=1 day)` spanning `[first start, last end)`.
    """
    return geoc.TimeDescriptor(
        bounds=geoc.TimeInterval(start=start_ms, end=end_ms),
        dimension=geoc.TimeDimension(
            actual_instance=geoc.Regular(
                origin=start_ms,
                step=geoc.TimeStep(granularity="days", step=1),
                type="regular",
            )
        ),
    )


def declare_md_dataset(variable, files):
    """State the metadata by hand instead of probing the files.

    One row per yearly file, covering all of that file's z slices. `leadingPrefix=[]` means
    a 3D array; a 4D `(time, depth, lat, lon)` file would use `[2]` to read depth 2.
    """
    config = VARIABLE_CONFIGS[variable]
    measurement = geoc.Measurement(
        actual_instance=geoc.ContinuousMeasurement(
            type="continuous", measurement=config["long_name"], unit=config["unit"]
        )
    )

    tiles = []
    for i, key in enumerate(files):
        year = file_year(key)
        steps = file_time_steps(year)
        tiles.append(
            geoc.AddDatasetMdTile(
                spatial_partition=PRESENTED_FOOTPRINT,
                band=0,
                z_index=i,
                array_name=variable,
                arrayGroup=None,
                leadingPrefix=[],
                time_descriptor=daily_descriptor(steps[0].start, steps[-1].end),
                time_steps=steps,
                params=geoc.GdalDatasetParameters(
                    file_path=vsicurl_url(key),
                    rasterband_channel=1,
                    geo_transform=STORED_GEO_TRANSFORM,
                    width=N_LON,
                    height=N_LAT,
                    file_not_found_handling="NoData",
                    # `None` is not "unknown": the reader falls back to the file's own
                    # `_FillValue` (1e20 in these files), which is what the probe stored.
                    noDataValue=None,
                    properties_mapping=None,
                    gdal_open_options=None,
                    gdal_config_options=[
                        ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS", VSICURL_ALLOWED_EXTENSIONS]
                    ],
                    allow_alphaband_as_mask=False,
                ),
            )
        )

    # the dataset's axis is the concatenation of the files' own axes
    first_start = tiles[0].time_steps[0].start
    last_end = tiles[-1].time_steps[-1].end

    meta_data = geoc.MetaDataDefinition(
        actual_instance=geoc.GdalMdMetaData(
            type="GdalMdMetaData",
            result_descriptor=geoc.RasterResultDescriptor(
                data_type="F32",
                spatial_reference="EPSG:4326",
                time=daily_descriptor(first_start, last_end),
                spatial_grid=geoc.SpatialGridDescriptor(
                    spatialGrid=geoc.SpatialGridDefinition(
                        geoTransform=PRESENTED_GEO_TRANSFORM,
                        gridBounds=geoc.GridBoundingBox2D(
                            topLeftIdx=geoc.GridIdx2D(xIdx=0, yIdx=0),
                            bottomRightIdx=geoc.GridIdx2D(xIdx=N_LON - 1, yIdx=N_LAT - 1),
                        ),
                    ),
                    descriptor="source",
                ),
                bands=[geoc.RasterBandDescriptor(name=config["long_name"], measurement=measurement)],
            ),
            z_role="variable",
            wrap=True,
            maxZBatchSize=MAX_Z_BATCH_SIZE,
            cacheTtl=None,
        )
    )
    return meta_data, tiles


In [ ]:
def register_dataset(model_id, scenario, variable, files):
    """State the metadata and register it as an `MdGdalSource` dataset."""
    config = VARIABLE_CONFIGS[variable]
    dataset_name = f"nexgddp_cmip6_{model_id}_{scenario}_{variable}"

    meta_data, tiles = declare_md_dataset(variable, files)
    md = meta_data.actual_instance
    bounds = md.result_descriptor.time.bounds
    print(
        f"  declared: {len(tiles)} files, z_role={md.z_role}, wrap={md.wrap}, "
        f"maxZBatchSize={md.max_z_batch_size}, "
        f"time {dt.datetime.fromtimestamp(bounds.start / 1000, tz=None).date()} .. "
        f"{dt.datetime.fromtimestamp(bounds.end / 1000, tz=None).date()}, "
        f"bands {[b.name for b in md.result_descriptor.bands]}"
    )

    properties = ge.AddDatasetProperties(
        name=dataset_name,
        display_name=f"NEX-GDDP-CMIP6 {config['name']} ({scenario})",
        description=(
            f"Daily NEX-GDDP-CMIP6 {config['name']} projections for {config['measurement']} "
            f"({config['unit']}). Model: {model_id}, Scenario: {scenario}, "
            f"one MD array file per year read through GDAL's multidimensional API"
        ),
        source_operator="MdGdalSource",
        symbology=build_symbology(variable),
    )

    dataset = ge.add_or_replace_dataset_with_permissions(
        DATA_STORE,
        properties,
        meta_data,
        permission_tuples=permission_tuples,
        replace_existing=True,
        timeout=600,
    )

    ge.add_md_dataset_tiles(dataset, tiles, timeout=600)
    print(f"  registered: {dataset}")
    return dataset


# Register all datasets
registered_datasets = []

for model, scenario, variable, files in TARGETS:
    print(f"\nProcessing: {model['id']} / {scenario} / {variable}")
    registered_datasets.append(register_dataset(model["id"], scenario, variable, files))

print(f"\nTotal datasets: {len(registered_datasets)}")


## Verification

Read one window per registered dataset, straddling a year boundary. Since the metadata is
stated rather than derived, this also has to check that the statement *landed* - so the
metadata round-trip is asserted before any pixel is read.

Three things are checked, in order:

1. **The stated metadata is what the server stored.** `z_role`, `wrap`, `maxZBatchSize`, the
   band name, and the dataset's time bounds `[noon(Y_first), noon(Y_last+1))`. A wrong axis
   length or a wrong band shows up here, without reading a pixel.
2. **The window straddles a file boundary** - file 1 ends and file 2 begins at the first
   year boundary - and `READ_DAYS > MAX_Z_BATCH_SIZE`, so the crossing is split across worker
   requests too. The days must be consecutive, one step each, on one band, and tiled.
3. **The stamps are the stated ones.** Each returned tile's `time_start_ms` must equal the
   noon-based stamp computed above, not merely the right date: a 12-hour drift (midnight
   intervals instead of noon) is exactly the mistake that would otherwise go unnoticed.

Values are then checked for being real data: inside the configured range, varying, and
identical when the same window is read twice.


In [ ]:
READ_DAYS = 12  # > MAX_Z_BATCH_SIZE, so the read spans more than one worker request


def first_boundary_window(time_bounds):
    """A [start, end) window centred on the first year boundary inside the dataset."""
    start = dt.datetime.fromtimestamp(time_bounds.start / 1000, tz=dt.timezone.utc)
    boundary = start.replace(year=start.year + 1)  # file 1 -> file 2
    half = dt.timedelta(days=READ_DAYS // 2)
    return boundary - half, boundary + half


async def read_window(workflow, window_start, window_end):
    """Stream a global window and group the tiles by day."""
    bbox = ge.QueryRectangle(
        ge.BoundingBox2D(-180.0, -90.0, 180.0, 90.0),
        ge.TimeInterval(window_start, window_end),
    )
    per_day = {}
    async for tile in workflow.raster_stream(bbox, open_timeout=600):
        day = str(tile.time.start)[:10]
        per_day.setdefault(day, []).append(tile)
    return per_day


def stamp_ms(tile):
    """A tile's time stamp in ms (`time_start_ms` is a numpy datetime64)."""
    return int(np.datetime64(tile.time_start_ms, "ms").astype("int64"))


async def verify_dataset(dataset_name, variable, files):
    """Assert the stated metadata landed, then read a boundary-crossing window."""
    config = VARIABLE_CONFIGS[variable]
    info = ge.dataset_info_by_name(dataset_name)
    assert info is not None, f"{dataset_name} is not registered"

    # `result_descriptor` is a oneOf wrapper; the concrete descriptor is under
    # `actual_instance`
    rd = info.result_descriptor.actual_instance
    print(f"\n{dataset_name}")
    print(f"  data_type={rd.data_type} bands={[b.name for b in rd.bands]} srs={rd.spatial_reference}")

    # --- 1. the stated metadata is what the server stored -------------------
    #
    # The dataset listing exposes `result_descriptor` only: `zRole`, `wrap` and
    # `maxZBatchSize` are dataset-internal and cannot be read back. They are still exercised
    # by the read below - `wrap` is why the values land in the presented -180..180 frame,
    # `zRole` is why a time axis exists at all, and `maxZBatchSize` is why a 12-day window
    # is split across worker requests.
    assert [b.name for b in rd.bands] == [config["long_name"]], (
        f"band is {[b.name for b in rd.bands]}, expected {[config['long_name']]}"
    )
    assert rd.spatial_reference == "EPSG:4326", rd.spatial_reference
    assert rd.data_type == "F32", rd.data_type
    gb = rd.spatial_grid.spatial_grid.grid_bounds
    assert (gb.top_left_idx.x_idx, gb.top_left_idx.y_idx) == (0, 0), gb
    assert (gb.bottom_right_idx.x_idx, gb.bottom_right_idx.y_idx) == (N_LON - 1, N_LAT - 1), (
        f"grid bounds {gb} do not describe a {N_LON}x{N_LAT} dataset"
    )

    first_year, last_year = file_year(files[0]), file_year(files[-1])
    assert rd.time.bounds.start == year_slice_start_ms(first_year), (
        f"axis starts at {rd.time.bounds.start}, expected noon of {first_year}-01-01 "
        f"({year_slice_start_ms(first_year)})"
    )
    assert rd.time.bounds.end == year_slice_start_ms(last_year + 1), (
        f"axis ends at {rd.time.bounds.end}, expected noon of {last_year + 1}-01-01"
    )
    n_declared = sum(len(file_time_steps(y)) for y in range(first_year, last_year + 1))
    print(f"  axis: {n_declared} daily slices, "
          f"{dt.date(first_year, 1, 1)} .. {dt.date(last_year, 12, 31)} (noon to noon)")

    # the workflow is assembled from typed operators, exactly like the Rust API expects it
    workflow = ge.register_workflow(
        workflow=ge.workflow_builder.operators.MdGdalSource(dataset=dataset_name)
    )

    # --- 2. a window straddling the first file boundary --------------------
    window_start, window_end = first_boundary_window(rd.time.bounds)
    print(f"  reading {window_start.date()} .. {window_end.date()} (crosses a year boundary)")

    per_day = await read_window(workflow, window_start, window_end)

    expected_days = (window_end - window_start).days
    assert len(per_day) == expected_days, (
        f"expected {expected_days} daily steps, got {len(per_day)}: {sorted(per_day)}"
    )
    days = sorted(per_day)
    assert days[0] == window_start.date().isoformat(), (
        f"first step is {days[0]}, expected {window_start.date()}"
    )
    parsed = [dt.date.fromisoformat(d) for d in days]
    assert all((b - a).days == 1 for a, b in zip(parsed, parsed[1:])), f"gap in {days}"

    tiles_per_day = {len(v) for v in per_day.values()}
    assert len(tiles_per_day) == 1, f"inconsistent tiling per day: {tiles_per_day}"
    n_tiles = tiles_per_day.pop()
    assert n_tiles > 1, "a global 0.25-degree query must be tiled, got a single tile"

    bands = {t.band for tiles in per_day.values() for t in tiles}
    assert bands == {0}, f"{variable} is single-band, got bands {bands}"

    # --- 3. the stamps are the stated noon-to-noon ones --------------------
    for day, date in zip(days, parsed):
        expected_ms = (
            year_slice_start_ms(date.year)
            + (date - dt.date(date.year, 1, 1)).days * DAY_MS
        )
        for t in per_day[day]:
            assert stamp_ms(t) == expected_ms, (
                f"{day}: tile stamped {stamp_ms(t)}, expected noon {expected_ms}"
            )
            assert stamp_ms_end(t) - expected_ms == DAY_MS, (
                f"{day}: tile does not span exactly one day"
            )
    print(f"  stamps: all {len(days)} days start at the stated noon")

    # --- values: inside the range the symbology was configured for -----------
    values = np.concatenate(
        [np.asarray(t.data, dtype="float64").ravel() for tiles in per_day.values() for t in tiles]
    )
    finite = values[np.isfinite(values)]
    assert finite.size, "no finite pixels at all -- the read returned only nodata"

    lo, hi = float(finite.min()), float(finite.max())
    print(f"  {len(per_day)} days x {n_tiles} tiles, band {bands}")
    print(f"  value range={lo:.2f}..{hi:.2f} {config['unit']} "
          f"(symbology {config['min']}..{config['max']})")

    # Not all returned pixels are data, and the two reasons are unrelated - so they are
    # reported separately rather than as one "padding" ratio that looks alarming but is not.
    #
    #  * Tiles are anchored at world (0,0) with a 512 px tile size (Geo Engine's tiling
    #    origin reference), while this dataset's presented grid is -180..180 x 60S..90N. So the
    #    tiles cover a larger area than the data and part of every tile is structural padding.
    #  * NEX-GDDP-CMIP6 itself masks a large part of the globe with a nodata sentinel, and the
    #    reader turns nodata into NaN. That is data, not plumbing.
    #
    # Structural coverage is therefore the dataset area over the area the tiles span:
    gb = rd.spatial_grid.spatial_grid.grid_bounds
    tl, br = gb.top_left_idx, gb.bottom_right_idx
    data_px = (br.y_idx - tl.y_idx + 1) * (br.x_idx - tl.x_idx + 1)
    tile_px = values.size // len(per_day)          # the tile block the query returns per day
    structural = data_px / tile_px
    nodata = finite.size / (len(per_day) * data_px)
    print(f"  tiles span {tile_px / data_px:.2f}x the dataset area "
          f"-> structural coverage {structural:.1%}")
    print(f"  of the dataset area {nodata:.1%} carries data "
          f"-> {structural * nodata:.1%} of all returned pixels are values")

    # generous bounds: this asserts the read is real data, not that it is bit-exact
    assert config["min"] <= lo and hi <= config["max"], (
        f"values {lo:.2f}..{hi:.2f} outside the plausible {config['min']}..{config['max']} "
        f"for {variable}"
    )
    # a global field should vary, not be a constant fill
    assert hi - lo > 1.0, f"suspiciously flat field: {lo:.2f}..{hi:.2f}"

    # --- determinism: the same window twice must agree (exercises the cache hint) ---
    again = await read_window(workflow, window_start, window_end)
    assert sorted(again) == days, "a repeated read returned different days"
    for day in days:
        a = np.sort(np.concatenate([np.asarray(t.data, dtype="float64").ravel() for t in per_day[day]]))
        b = np.sort(np.concatenate([np.asarray(t.data, dtype="float64").ravel() for t in again[day]]))
        assert np.array_equal(a, b, equal_nan=True), f"repeated read of {day} differs"

    print("  OK")
    return per_day


def stamp_ms_end(tile):
    return int(np.datetime64(tile.time_end_ms, "ms").astype("int64"))


results = {}
for (model, scenario, variable, files), dataset_name in zip(TARGETS, registered_datasets):
    results[(scenario, variable)] = await verify_dataset(dataset_name, variable, files)

print("\nVerified:", ", ".join(f"{s}/{v} ({len(d)} days)" for (s, v), d in results.items()))


## Next Steps

1. Set `DEV_ONLY = False` to register the full matrix (5 models x 3 scenarios x 3 variables)
2. Query a registered layer through WMS / a temporal workflow and confirm daily slices render
3. Test the climate-risk process with the new datasets

Comparing with `nexgddp_cmip6_ingest.ipynb` (the probe version) shows what the probe saves.
The probe branch also fixes the single-array band name, so the two agree except for that
until it lands.
